In [ ]:
import math
import torchvision.transforms as transforms
from torchvision.utils import save_image
import PIL as pil
import os
import cv2
import torchvision.transforms.functional as F
import numpy as np
import torchvision
import torch
from tqdm import tqdm


In [ ]:
IMG_SIZE = 256
IMG_DIR = "data/"
BATCH_SIZE = 32 
NUM_WORKERS = os.cpu_count()-2

In [ ]:
class BenGrahamPreprocessing:
    def __init__(self, img_size):
        self.img_size = img_size

    def __call__(self, image: pil.Image.Image) -> pil.Image.Image:
        image_np = np.array(image)

        # Aplicar el desenfoque gaussiano de OpenCV
        blurred_image = cv2.GaussianBlur(image_np, (0, 0), self.img_size / 10)

        # Combinar la imagen original y la desenfocada
        combined_image = cv2.addWeighted(image_np, 4, blurred_image, -4, 128)

        # Convertir a escala de grises
        return F.to_pil_image(combined_image)

In [ ]:
class BackgroundRemover:
    def __init__(self):
        pass
    def __call__(self, image: pil.Image.Image, version=1) -> pil.Image.Image:
        self.ancho, self.alto = image.size
        if version==1:
            return self.recortar_bordes_negros(image)
        else:
            return self.recortar_bordes_negros_v2(image)
        
    def encontrar_limites(self, img):
        pixeles = img.load()
        
        ancho, alto = self.ancho, self.alto
        ancho_medio = ancho//2
        alto_medio = alto//2
        x_inicio = 0
        x_inicio = 0
        for x in range(ancho):
            if pixeles[x, alto_medio] != (0, 0, 0):
                x_inicio = x
                break

        x_fin = ancho - 1
        for x in range(ancho - 1, -1, -1):
            if pixeles[x, alto_medio] != (0, 0, 0): 
                x_fin = x
                break
        
        y_inicio = 0
        for y in range(alto):
            if pixeles[ancho_medio, y] != (0, 0, 0): 
                y_inicio = y
                break

        y_fin = alto - 1
        for y in range(alto - 1, -1, -1):
            if pixeles[ancho_medio, y] != (0, 0, 0):
                y_fin = y
                break
        
        return x_inicio, x_fin, y_inicio, y_fin
    
    def calcular_centro_y_radio(self, x_inicio, x_fin, y_inicio, y_fin):
        centro_x = (x_inicio + x_fin) // 2
        centro_y = (y_inicio + y_fin) // 2
        radio = max((x_fin - x_inicio) // 2, (y_fin - y_inicio) // 2)
        return centro_x, centro_y, radio
    
    def recortar_ojo(self, imagen, centro_x, centro_y, radio):
        lado = int((2 * radio) / math.sqrt(2))
        mitad_lado = lado // 2
        
        izquierda = max(centro_x - mitad_lado, 0)
        arriba = max(centro_y - mitad_lado, 0)
        derecha = min(centro_x + mitad_lado, imagen.width)
        abajo = min(centro_y + mitad_lado, imagen.height)
        
        imagen_recortada = imagen.crop((izquierda, arriba, derecha, abajo))
        return imagen_recortada
    
    def recortar_bordes_negros(self, imagen):
        x_inicio, x_fin, y_inicio, y_fin = self.encontrar_limites(imagen)
        
        imagen_recortada = imagen.crop((x_inicio, y_inicio, x_fin + 1, y_fin + 1))
        return imagen_recortada
    
    def recortar_bordes_negros_v2(self, img):
        x_inicio, x_fin, y_inicio, y_fin = self.encontrar_limites(img)
            
        centro_x, centro_y, radio = self.calcular_centro_y_radio(x_inicio, x_fin, y_inicio, y_fin)
            
        return self.recortar_ojo(img, centro_x, centro_y, radio)

In [ ]:
transform = transforms.Compose(
    [
        BackgroundRemover(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        #  transforms.Grayscale(),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        # BenGrahamPreprocessing(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225]),
    ]
)

In [ ]:
# Load the training data from the data folder
trainset = torchvision.datasets.ImageFolder(root="data/train/", transform=transform)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
testset = torchvision.datasets.ImageFolder(root="data/test/", transform=transform)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)

In [ ]:

output_folder = "data/transformed_images"
os.makedirs(output_folder, exist_ok=True)

for i, (image, label) in tqdm(enumerate(trainloader)):
    for j in range(image.size(0)):
        save_image(image[j], os.path.join(output_folder, f"image_{i * BATCH_SIZE + j}.png"))

print(f"Transformed images saved to '{output_folder}'.")
